In [13]:
import sys
import os
import torch
from transformers import pipeline

# Add parent directory to path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

print(f"System ready. GPU Available: {torch.cuda.is_available()}")

System ready. GPU Available: True


In [14]:
device = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    "zero-shot-classification", 
    model="facebook/bart-large-mnli", 
    device=device
)


In [15]:

test_sentences = [
    # TYPE 1: FACTS (Targets)
    "The unemployment rate in Singapore dropped to 2.1% last quarter.",
    "More than 500,000 people attended the rally in Jakarta.",
    "Scientists have discovered a new variant of the virus.",
    
    # TYPE 2: OPINIONS (Ambiguous - usually drop)
    "I believe the government is doing a terrible job.",
    "The scenery was absolutely breathtaking.",
    "It is vital that we consider the implications of this policy.",
    
    # TYPE 3: NOISE (Drop)
    "Hello and welcome to the channel.",
    "Please subscribe for more updates.",
    "As shown in the graph below."
]



In [16]:
labels = [
    "a specific verifiable fact or event",   
    "a subjective opinion or interpretation", 
    "generic news structure or headline",    
    "future prediction or speculation"       
]

hypothesis_template = "The text contains {}."

def test_labels(labels, sentences):
    print(f"\n--- Testing Labels ---")
    print(f"{'SENTENCE':<60} | {'PREDICTION':<35} | {'SCORE'}")
    print("-" * 110)
    
    for sent in sentences:
        result = classifier(
            sent, 
            labels, 
            hypothesis_template=hypothesis_template
        )
        
        top_label = result['labels'][0]
        top_score = result['scores'][0]
        
        # Truncate
        disp = (sent[:57] + '...') if len(sent) > 57 else sent
        
        # Color coding for easier reading
        if top_label == "a specific verifiable fact or event":
            label_display = f"✅ {top_label}" 
        else:
            label_display = f"❌ {top_label}"
            
        print(f"{disp:<60} | {label_display:<35} | {top_score:.2f}")

# Run with the problematic sentences
problematic_sentences = [
    "Breaking news from the capital.",                   # Should be Structure
    "This is a bold move that will surely upset the opposition.", # Should be Opinion
    "The project is expected to complete by 2030.",      # Should be Fact (Date)
    "The Prime Minister announced a $5 billion package." # Should be Fact (Number)
]

test_labels(labels, problematic_sentences)


--- Testing Labels ---
SENTENCE                                                     | PREDICTION                          | SCORE
--------------------------------------------------------------------------------------------------------------
Breaking news from the capital.                              | ❌ future prediction or speculation  | 0.39
This is a bold move that will surely upset the opposition... | ❌ future prediction or speculation  | 0.51
The project is expected to complete by 2030.                 | ❌ future prediction or speculation  | 0.47
The Prime Minister announced a $5 billion package.           | ✅ a specific verifiable fact or event | 0.67


In [17]:
def extract_checkworthy_claims(sentences, threshold=0.7):
    """
    Refined logic using specific verifiability labels.
    """
    # These labels separated the signal from noise best in our test
    labels = [
        "a specific verifiable fact or event", 
        "a subjective opinion or interpretation", 
        "generic news structure or headline",
        "future prediction or speculation"
    ]
    
    kept_claims = []
    dropped_logs = []
    
    hypothesis_template = "The text contains {}."
    
    for sent in sentences:
        result = classifier(
            sent, 
            labels,
            hypothesis_template=hypothesis_template
        )
        
        top_label = result['labels'][0]
        score = result['scores'][0]
        
        # LOGIC: Only keep specific verifiable facts
        if top_label == "a specific verifiable fact or event" and score > threshold:
            kept_claims.append({"text": sent, "score": score})
        else:
            dropped_logs.append({"text": sent, "label": top_label, "score": score})
            
    return kept_claims, dropped_logs


In [18]:
# Simulating a mixed paragraph from a news scraper
raw_text = [
    "Breaking news from the capital.",
    "The Prime Minister announced a $5 billion infrastructure package.",
    "This is a bold move that will surely upset the opposition.",
    "Critics say the funding is insufficient.",
    "The project is expected to complete by 2030."
]

print("running extraction...")
claims, dropped = extract_checkworthy_claims(raw_text, threshold=0.6)

print(f"\nRETAINED CLAIMS ({len(claims)}):")
for c in claims:
    print(f"[{c['score']:.2f}] {c['text']}")

print(f"\nDROPPED ({len(dropped)}):")
for d in dropped:
    print(f"[{d['label']} - {d['score']:.2f}] {d['text']}")

running extraction...

RETAINED CLAIMS (1):
[0.69] The Prime Minister announced a $5 billion infrastructure package.

DROPPED (4):
[future prediction or speculation - 0.39] Breaking news from the capital.
[future prediction or speculation - 0.51] This is a bold move that will surely upset the opposition.
[a subjective opinion or interpretation - 0.55] Critics say the funding is insufficient.
[future prediction or speculation - 0.47] The project is expected to complete by 2030.
